## Cleaned complete cells

## Rainfall Prediction

In [12]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import joblib
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
COLOMBO_LAT = 6.9271
COLOMBO_LON  = 79.8612

# ── Load models ───────────────────────────────────────────────────────────────
stage1_model = joblib.load('rainfall/stage1_rain_classifier.pkl')
stage2_model = joblib.load('rainfall/stage2_amount_regressor.pkl')
FEATURE_NAMES = stage1_model.get_booster().feature_names

# ── Fetch weather data ────────────────────────────────────────────────────────
current_time = datetime.now()
current_date = current_time.date()
hourly_vars  = ["temperature_2m", "relative_humidity_2m", "dew_point_2m",
                "precipitation", "surface_pressure", "cloud_cover",
                "wind_speed_10m", "wind_direction_10m"]

def fetch_df(url, params):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()['hourly']
    return pd.DataFrame({'datetime': pd.to_datetime(data['time']),
                         **{v: data[v] for v in hourly_vars}})

df_hist = fetch_df("https://archive-api.open-meteo.com/v1/archive", {
    "latitude": COLOMBO_LAT, "longitude": COLOMBO_LON,
    "start_date": str(current_date - timedelta(days=20)),
    "end_date":   str(current_date - timedelta(days=4)),
    "hourly": ",".join(hourly_vars), "timezone": "Asia/Colombo"
})

df_recent = fetch_df("https://api.open-meteo.com/v1/forecast", {
    "latitude": COLOMBO_LAT, "longitude": COLOMBO_LON,
    "hourly": ",".join(hourly_vars),
    "past_days": 4, "forecast_days": 1, "timezone": "Asia/Colombo"
})
df_recent = df_recent[df_recent['datetime'] <= current_time]

df = (pd.concat([df_hist, df_recent], ignore_index=True)
        .drop_duplicates(subset=['datetime'], keep='last')
        .sort_values('datetime').reset_index(drop=True))

# ── Feature engineering ───────────────────────────────────────────────────────
df['hour']        = df['datetime'].dt.hour
df['month']       = df['datetime'].dt.month
df['year']        = df['datetime'].dt.year
df['day_of_year'] = df['datetime'].dt.dayofyear
df['day_of_week'] = df['datetime'].dt.dayofweek
df['date_dt']     = df['datetime'].dt.normalize()

W, MP = 24, 18

df['precip_24h_sum']              = df['precipitation'].rolling(W, min_periods=MP).sum()
df['precip_24h_max']              = df['precipitation'].rolling(W, min_periods=MP).max()
df['precip_24h_mean']             = df['precipitation'].rolling(W, min_periods=MP).mean()
df['precip_24h_std']              = df['precipitation'].rolling(W, min_periods=MP).std()
df['precip_24h_rainy_hours']      = df['precipitation'].rolling(W, min_periods=MP).apply(lambda x: (x > 0.1).sum(), raw=True)

df['temp_24h_mean']  = df['temperature_2m'].rolling(W, min_periods=MP).mean()
df['temp_24h_max']   = df['temperature_2m'].rolling(W, min_periods=MP).max()
df['temp_24h_min']   = df['temperature_2m'].rolling(W, min_periods=MP).min()
df['temp_24h_range'] = df['temp_24h_max'] - df['temp_24h_min']
df['temp_24h_std']   = df['temperature_2m'].rolling(W, min_periods=MP).std()
df['temp_24h_trend'] = df['temperature_2m'] - df['temperature_2m'].shift(W)

df['humidity_24h_mean']           = df['relative_humidity_2m'].rolling(W, min_periods=MP).mean()
df['humidity_24h_max']            = df['relative_humidity_2m'].rolling(W, min_periods=MP).max()
df['humidity_24h_min']            = df['relative_humidity_2m'].rolling(W, min_periods=MP).min()
df['humidity_24h_range']          = df['humidity_24h_max'] - df['humidity_24h_min']
df['humidity_24h_hours_above_80'] = df['relative_humidity_2m'].rolling(W, min_periods=MP).apply(lambda x: (x > 80).sum(), raw=True)

df['pressure_24h_mean']  = df['surface_pressure'].rolling(W, min_periods=MP).mean()
df['pressure_24h_min']   = df['surface_pressure'].rolling(W, min_periods=MP).min()
df['pressure_24h_max']   = df['surface_pressure'].rolling(W, min_periods=MP).max()
df['pressure_24h_std']   = df['surface_pressure'].rolling(W, min_periods=MP).std()
df['pressure_24h_trend'] = df['surface_pressure'] - df['surface_pressure'].shift(W)

df['wind_24h_mean'] = df['wind_speed_10m'].rolling(W, min_periods=MP).mean()
df['wind_24h_max']  = df['wind_speed_10m'].rolling(W, min_periods=MP).max()
df['wind_24h_std']  = df['wind_speed_10m'].rolling(W, min_periods=MP).std()

df['cloud_24h_mean']              = df['cloud_cover'].rolling(W, min_periods=MP).mean()
df['cloud_24h_max']               = df['cloud_cover'].rolling(W, min_periods=MP).max()
df['cloud_24h_hours_above_90']    = df['cloud_cover'].rolling(W, min_periods=MP).apply(lambda x: (x > 90).sum(), raw=True)

df['dewpoint_24h_mean'] = df['dew_point_2m'].rolling(W, min_periods=MP).mean()

df['temp_current']                   = df['temperature_2m']
df['humidity_current']               = df['relative_humidity_2m']
df['pressure_current']               = df['surface_pressure']
df['dewpoint_current']               = df['dew_point_2m']
df['wind_speed_current']             = df['wind_speed_10m']
df['wind_direction_current']         = df['wind_direction_10m']
df['cloud_cover_current']            = df['cloud_cover']
df['dew_point_depression_current']   = df['temperature_2m'] - df['dew_point_2m']
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

daily_agg = df.groupby('date_dt').agg(
    daily_rainfall_total=('precipitation', 'sum'),
    daily_temp_mean=('temperature_2m', 'mean'),
    daily_pressure_mean=('surface_pressure', 'mean'),
    daily_humidity_mean=('relative_humidity_2m', 'mean'),
).reset_index().sort_values('date_dt').reset_index(drop=True)

for lag in [1, 2, 3, 7, 14]:
    daily_agg[f'rainfall_lag_{lag}d'] = daily_agg['daily_rainfall_total'].shift(lag)
for lag in [1, 3, 7]:
    daily_agg[f'temp_lag_{lag}d'] = daily_agg['daily_temp_mean'].shift(lag)
for lag in [1, 3]:
    daily_agg[f'pressure_lag_{lag}d'] = daily_agg['daily_pressure_mean'].shift(lag)
for lag in [1, 3]:
    daily_agg[f'humidity_lag_{lag}d'] = daily_agg['daily_humidity_mean'].shift(lag)

lag_cols = [c for c in daily_agg.columns if '_lag_' in c]
df = df.merge(daily_agg[['date_dt'] + lag_cols], on='date_dt', how='left')

df['month_sin']       = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']       = np.cos(2 * np.pi * df['month'] / 12)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

df['season_SW_Monsoon']  = df['month'].isin([5, 6, 7]).astype(int)
df['season_NE_Monsoon']  = df['month'].isin([12, 1, 2]).astype(int)
df['season_Inter_Heavy'] = df['month'].isin([4, 10, 11]).astype(int)
df['season_Inter_Light'] = (~df['month'].isin([5,6,7,12,1,2,4,10,11])).astype(int)

df['days_into_sw_monsoon'] = df.apply(
    lambda r: (r['datetime'] - pd.Timestamp(year=r['year'], month=5, day=1)).days
              if r['month'] in [5, 6, 7] else 0, axis=1)
df['days_into_ne_monsoon'] = df.apply(
    lambda r: (r['datetime'] - pd.Timestamp(
                   year=r['year'] if r['month'] == 12 else r['year'] - 1, month=12, day=1)).days
              if r['month'] in [12, 1, 2] else 0, axis=1)

df['pressure_tendency_3h']    = (df['surface_pressure'] - df['surface_pressure'].shift(3)) / 3
df['pressure_tendency_6h']    = (df['surface_pressure'] - df['surface_pressure'].shift(6)) / 6
df['wind_from_southwest']     = ((df['wind_direction_10m'] >= 180) & (df['wind_direction_10m'] <= 270)).astype(int)
df['wind_from_northeast']     = ((df['wind_direction_10m'] >= 0)   & (df['wind_direction_10m'] <= 90)).astype(int)
raw_change = np.abs(df['wind_direction_10m'] - df['wind_direction_10m'].shift(6))
df['wind_direction_change_6h'] = raw_change.apply(lambda x: min(x, 360 - x) if pd.notna(x) else x)
df['instability_index']       = (df['temp_24h_range'] * (1 - df['dew_point_depression_current'] / 20)).clip(lower=0)
df['moisture_flux']           = df['humidity_24h_mean'] * df['wind_24h_mean']

# ── Build prediction vector ───────────────────────────────────────────────────
latest = df.iloc[-1:]
X_pred = pd.DataFrame(index=[0], columns=FEATURE_NAMES, dtype=float)
for feat in FEATURE_NAMES:
    X_pred[feat] = latest[feat].values[0] if feat in latest.columns else 0.0
X_pred = X_pred.fillna(0).astype(float)

# ── Predict ───────────────────────────────────────────────────────────────────
rain_probability = stage1_model.predict_proba(X_pred)[0, 1]
predicted_amount = max(0, stage2_model.predict(X_pred)[0])
final_prediction = max(0, rain_probability * predicted_amount)

print(f"{final_prediction:.2f}")


2.77


## ETO Calculation

In [ ]:
import math
import requests
from datetime import date

# --- FAO-56 Helper Functions ---

def saturation_vapor_pressure(T):
    return 0.6108 * math.exp((17.27 * T) / (T + 237.3))

def slope_svp_curve(T):
    return 4098.0 * saturation_vapor_pressure(T) / ((T + 237.3) ** 2)

def actual_vapor_pressure(RHmin, RHmax, es_min, es_max):
    return ((RHmax / 100.0) * es_min + (RHmin / 100.0) * es_max) / 2.0

def psychrometric_constant(elevation_m):
    P = 101.3 * ((293.0 - 0.0065 * elevation_m) / 293.0) ** 5.26
    return 0.000665 * P

def wind_speed_10m_to_2m(u10):
    return u10 * 4.87 / math.log(67.8 * 10.0 - 5.42)

def extraterrestrial_radiation(day_of_year, latitude_deg):
    phi = math.radians(latitude_deg)
    dr = 1 + 0.033 * math.cos((2 * math.pi / 365) * day_of_year)
    delta = 0.409 * math.sin((2 * math.pi / 365) * day_of_year - 1.39)
    ws = math.acos(-math.tan(phi) * math.tan(delta))
    return (24 * 60 / math.pi) * 0.0820 * dr * (
        ws * math.sin(phi) * math.sin(delta) +
        math.cos(phi) * math.cos(delta) * math.sin(ws)
    )

def net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m):
    Rns = (1.0 - 0.23) * Rs
    Rso = (0.75 + 2e-5 * elevation_m) * Ra
    Rs_Rso = max(0.0, min(Rs / Rso if Rso > 0 else 0.0, 1.5))
    Tavg4 = ((Tmax + 273.16) ** 4 + (Tmin + 273.16) ** 4) / 2.0
    Rnl = 4.903e-9 * Tavg4 * (0.34 - 0.14 * math.sqrt(max(ea, 0.0))) * (1.35 * Rs_Rso - 0.35)
    return Rns - Rnl

def compute_daily_eto(Tmax, Tmin, RHmax, RHmin, u10_kmh, Rs, elevation_m, latitude_deg, day_of_year):
    Tmean = (Tmax + Tmin) / 2.0
    es = (saturation_vapor_pressure(Tmax) + saturation_vapor_pressure(Tmin)) / 2.0
    ea = actual_vapor_pressure(RHmin, RHmax, saturation_vapor_pressure(Tmin), saturation_vapor_pressure(Tmax))
    delta = slope_svp_curve(Tmean)
    gamma = psychrometric_constant(elevation_m)
    Ra = extraterrestrial_radiation(day_of_year, latitude_deg)
    Rn = net_radiation_fao(Rs, Tmax, Tmin, ea, Ra, elevation_m)
    u2 = wind_speed_10m_to_2m(u10_kmh / 3.6)
    numerator = 0.408 * delta * Rn + gamma * (900.0 / (Tmean + 273.0)) * u2 * (es - ea)
    denominator = delta + gamma * (1.0 + 0.34 * u2)
    return numerator / max(denominator, 1e-6)

# --- Fetch Today's Data ---
LATITUDE    = 6.9271
LONGITUDE   = 79.8612
ELEVATION_M = 7
today = date.today().isoformat()
doy   = date.today().timetuple().tm_yday

params = {
    "latitude": LATITUDE, "longitude": LONGITUDE,
    "start_date": today, "end_date": today,
    "daily": ["temperature_2m_max", "temperature_2m_min",
              "relative_humidity_2m_max", "relative_humidity_2m_min",
              "wind_speed_10m_mean", "shortwave_radiation_sum"],
    "timezone": "Asia/Colombo"
}

response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)
response.raise_for_status()
data = response.json()["daily"]

# --- Compute ETo ---
eto = compute_daily_eto(
    Tmax        = data["temperature_2m_max"][0],
    Tmin        = data["temperature_2m_min"][0],
    RHmax       = data["relative_humidity_2m_max"][0],
    RHmin       = data["relative_humidity_2m_min"][0],
    u10_kmh     = data["wind_speed_10m_mean"][0],
    Rs          = data["shortwave_radiation_sum"][0],
    elevation_m = ELEVATION_M,
    latitude_deg= LATITUDE,
    day_of_year = doy
)

print(f"{eto:.2f} mm/day")
